# POC segmentation CT cardiaque — exécution dans Colab

Ce notebook exécute les scripts du dépôt GitHub sur le GPU de Colab. Les données vont dans ton Google Drive.

**Avant de commencer** : menu **Exécution → Modifier le type d'exécution → GPU T4**. Puis exécute les cellules une par une (▶).

## 1. Relier Drive et vérifier le GPU
Une fenêtre d'autorisation s'ouvre : accepte avec ton **compte Google personnel**. Tu dois voir **Tesla T4**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Récupérer le dépôt GitHub
Le dépôt GitHub `Cameronprr/poc-segmentation-ct` est copié dans Colab (cette copie disparaît à la fin de la session ; le dépôt sur GitHub reste la référence).

In [ ]:
PSEUDO = 'Cameronprr'
!rm -rf /content/poc-segmentation-ct
!git clone https://github.com/{PSEUDO}/poc-segmentation-ct.git /content/poc-segmentation-ct
%cd /content/poc-segmentation-ct
!ls

## 3. Installer les bibliothèques de l'étape
Pour l'étape 2, seule `nibabel` (lecture des fichiers NIfTI) est nécessaire. Les autres (MONAI, nnU-Net, TotalSegmentator) seront installées aux étapes suivantes.

In [ ]:
!pip -q install nibabel

## 4. Étape 2 — Télécharger MM-WHS dans Drive
À ne faire qu'une fois : les données restent ensuite dans ton Drive. `megatools` est un petit outil qui télécharge depuis le service MEGA. Compte plusieurs minutes.

In [ ]:
DONNEES = '/content/drive/MyDrive/poc_ct/mmwhs'
!mkdir -p "$DONNEES"
!apt-get -qq install megatools > /dev/null
!megadl --path "$DONNEES" "https://mega.nz/folder/UNMF2YYI#1cqJVzo4p_wESv9P_pc8uA"

# Décompresser d'éventuelles archives
import glob, zipfile, os
for z in glob.glob(DONNEES + '/**/*.zip', recursive=True):
    print('Décompression :', z)
    zipfile.ZipFile(z).extractall(os.path.dirname(z))

## 5. Étape 2 — Vérifier les données
Le script `scripts/02_verifier_donnees.py` contrôle que les 20 cas du découpage figé sont complets et affiche leurs caractéristiques. **Résultat attendu : CONFORME — 20 cas complets.**

In [ ]:
!python scripts/02_verifier_donnees.py --donnees "$DONNEES"

**Fin de l'étape 2.** Copie le tableau affiché (ou le message d'erreur) et envoie-le dans la conversation.

## 6. Étape 3 — Installer MONAI
MONAI est une bibliothèque d'imagerie médicale (chargement des scanners, transformations, métriques). Le script de l'étape 3 s'en sert pour lire les volumes et calculer les miniatures. À refaire à chaque nouvelle session Colab, comme les cellules 1 et 2.

In [ ]:
!pip -q install "monai>=1.3" nibabel

## 7. Étape 3 — Contrôle qualité détaillé et conversion au format nnU-Net
Le script `scripts/03_preparer.py` contrôle les 20 cas : forme, affine, espacement, orientation, étiquettes et doublons entre test et entraînement. Il écrit ensuite le rapport `rapport_qc_etape3.csv`. Si tout est conforme, il écrit aussi le jeu `Dataset101_MMWHS` dans `poc_ct/nnunet_raw`.

**Résultat attendu :** « Résultat du contrôle : CONFORME », puis 15 cas dans imagesTr/labelsTr et 5 dans imagesTs/labelsTs.

Si le script s'arrête, le dernier message indique pourquoi :
- **NON CONFORME** : rien n'a été écrit au format nnU-Net ;
- **existe déjà** : le jeu a déjà été écrit. Ne relance avec `--ecraser` que si tu veux vraiment le remplacer ;
- **ERREUR PENDANT LA CONVERSION** : le dossier est incomplet. Relance avec `--ecraser` une fois la cause corrigée.

Les lignes « QUASI-DOUBLON possible » ne bloquent pas : envoie-les dans la conversation pour qu'on les examine.

In [ ]:
DONNEES = '/content/drive/MyDrive/poc_ct/mmwhs'
SORTIE = '/content/drive/MyDrive/poc_ct/nnunet_raw'
!python scripts/03_preparer.py --donnees "$DONNEES" --sortie "$SORTIE"

**Fin de l'étape 3.** Copie tout ce que le script a affiché (tableau, ressemblances, avertissements, résultat) et envoie-le dans la conversation pour la vérification par l'agent QA.